# Attention Lab 2 · Scoring Functions

> **Types of Attention — Lab 2.** Run top to bottom (Runtime → Run all). Read the plain-English notes, run the code,
> then do the **Your turn 🧪** cell. Everything is built from scratch with NumPy so nothing is hidden.

### In plain English
Before a word splits its focus, the model rates **how well two words match**. There are two classic
recipes:

- **Dot-product (quick):** a fast similarity rating — like glancing at two dating profiles and rating how alike they are.
  Simple, but the scores balloon as word descriptions get longer.
- **Scaled dot-product:** the same thing, divided down so it stays sensible. **Every modern Transformer uses this.**
- **Additive (careful):** a tiny built-in judge weighs the two words together. Slower, but works even when the two things
  compared aren't the same size.

Whichever you use, the next step is identical: turn scores into percentages of focus.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
VIOLET, CYAN, AMBER = "#6C5CE7", "#22D3EE", "#F59E0B"
rng = np.random.default_rng(1)
def softmax(x):
    x = x - x.max(); e = np.exp(x); return e / e.sum()

## 1 · The three scoring functions

In [ ]:
def score_dot(q, K):                 # Luong / multiplicative
    return K @ q

def score_scaled(q, K):              # Transformer
    return (K @ q) / np.sqrt(q.shape[-1])

def score_additive(q, K, W_q, W_k, v):   # Bahdanau
    # a tiny neural net: v^T tanh(W_q q + W_k k)
    return np.tanh(K @ W_k.T + (W_q @ q)) @ v

d = 8
q = rng.normal(size=d); K = rng.normal(size=(6, d))
W_q = rng.normal(size=(d, d)) * .3; W_k = rng.normal(size=(d, d)) * .3; v = rng.normal(size=d)

for name, s in [("dot", score_dot(q, K)),
                ("scaled", score_scaled(q, K)),
                ("additive", score_additive(q, K, W_q, W_k, v))]:
    print(f"{name:9s} scores {np.round(s,2)}  ->  weights {np.round(softmax(s),2)}")

## 2 · The scaling problem, measured
As the key dimension `d` grows, raw dot products grow like √d. Softmax then saturates: attention stops blending and
picks one word. Scaling by 1/√d holds it steady.

In [ ]:
ds = [2, 4, 8, 16, 32, 64, 128, 256]
raw_max, scaled_max = [], []
for d in ds:
    q = rng.normal(size=d); K = rng.normal(size=(8, d))
    raw_max.append(softmax(score_dot(q, K)).max())
    scaled_max.append(softmax(score_scaled(q, K)).max())

plt.figure(figsize=(6.5,3.6))
plt.plot(ds, raw_max, "o-", color=AMBER, lw=2, label="dot (unscaled)")
plt.plot(ds, scaled_max, "o-", color=VIOLET, lw=2, label="scaled dot")
plt.axhline(1/8, color="#999", ls=":", label="uniform (1/8)")
plt.xscale("log", base=2); plt.xlabel("key dimension $d_k$"); plt.ylabel("largest attention weight")
plt.title("unscaled attention collapses onto one key"); plt.legend(); plt.tight_layout(); plt.show()

## 3 · Additive attention handles mismatched sizes
Dot-product needs `dim(q) == dim(k)`. Additive projects both first, so it doesn't.

In [ ]:
d_q, d_k, d_h = 4, 9, 6          # different query and key sizes!
q = rng.normal(size=d_q); K = rng.normal(size=(5, d_k))
W_q = rng.normal(size=(d_h, d_q))*.5; W_k = rng.normal(size=(d_h, d_k))*.5; v = rng.normal(size=d_h)
scores = np.tanh(K @ W_k.T + (W_q @ q)) @ v
print("query dim", d_q, "| key dim", d_k, "-> additive scores work fine:", np.round(scores, 2))
try:
    K @ q
except ValueError as e:
    print("dot-product fails:", e)

## Your turn 🧪
1. At `d=256`, how many keys does unscaled attention effectively use? (Compute `1 / (weights**2).sum()` — the
   "effective number of keys".)
2. Try scaling by `1/d` instead of `1/√d`. Do the weights become too *flat*?
3. Additive attention has learnable parameters (`W_q, W_k, v`); dot-product has none. Why does that make additive slower?

In [ ]:
# Your turn: effective number of keys
def eff_keys(w): return 1.0 / (w**2).sum()
d = 256
q = rng.normal(size=d); K = rng.normal(size=(8, d))
print("unscaled: uses ~%.2f of 8 keys" % eff_keys(softmax(score_dot(q, K))))
print("scaled:   uses ~%.2f of 8 keys" % eff_keys(softmax(score_scaled(q, K))))